In [ ]:
# Instala ttyd (terminal web) + cloudflared (tunnel publico)
!apt-get update -y
!apt-get install -y ttyd wget
!if [ ! -x /usr/bin/ttyd ]; then wget -q https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64 -O /usr/local/bin/ttyd && chmod +x /usr/local/bin/ttyd; fi
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!ttyd --version

In [ ]:
# Sobe terminal web sem senha e cria URL publica
import re
import subprocess

TERMINAL_PORT = 7681

ttyd_proc = subprocess.Popen(
    [
        "ttyd",
        "-p", str(TERMINAL_PORT),
        "bash", "-lc", "cd /content && exec bash"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{TERMINAL_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

terminal_url = None
for line in tunnel_proc.stdout:
    print(line.strip())
    m = re.search(r"(https://.*\.trycloudflare\.com)", line)
    if m:
        terminal_url = m.group(1)
        break

if not terminal_url:
    raise RuntimeError("Nao foi possivel criar URL publica do terminal")

print("\nTerminal web pronto (SEM senha):")
print(terminal_url)